In [1]:
!pip install -q transformers datasets torch accelerate pandas


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Shuvo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import torch 
import pandas as pd 
from datasets import load_dataset 
from transformers import AutoTokenizer, AutoModelForCausalLM 

In [5]:
MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

model.to("cpu")
model.eval()

print("Model loaded!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:01<00:00, 162.20it/s]


Model loaded!


In [6]:
dataset = load_dataset(
    "cais/mmlu",
    "abstract_algebra"
)

dataset

C:\Users\Shuvo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shuvo\.cache\huggingface\hub\datasets--cais--mmlu. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating dev split: 100%|█████████

DatasetDict({
    test: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 11
    })
    dev: Dataset({
        features: ['question', 'subject', 'choices', 'answer'],
        num_rows: 5
    })
})

In [7]:
LETTERS = ["A", "B", "C", "D"]

def make_prompt(example):
    question = example["question"]
    choices = example["choices"]

    prompt = f"""Answer the following multiple-choice question.

Question:
{question}

A. {choices[0]}
B. {choices[1]}
C. {choices[2]}
D. {choices[3]}

Return only one letter: A, B, C, or D.

Answer:"""

    return prompt

In [8]:
print(make_prompt(dataset["test"][0]))

Answer the following multiple-choice question.

Question:
Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.

A. 0
B. 4
C. 2
D. 6

Return only one letter: A, B, C, or D.

Answer:


In [9]:
def get_prediction(example):
    prompt = make_prompt(example)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer

In [10]:
prediction = get_prediction(test_data[0])

print("Model output:", repr(prediction))
print("Correct:", LETTERS[test_data[0]["answer"]])

NameError: name 'test_data' is not defined